# β2m DEPC Parser — Pipeline Demo

This notebook walks through the full pipeline on synthetic data.
It demonstrates every step from raw spectrum loading through Cu(II) protection analysis
and final array output.

**Run from the repository root.**

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

## 1. Generate synthetic test data

Simulates two LC-MS/MS runs: one without Cu(II), one with Cu(II).  
His residues are labeled ~60% (no Cu) and ~20% (Cu) by design.

In [ ]:
from pathlib import Path
import subprocess

raw_dir = Path('../data/raw')
raw_dir.mkdir(parents=True, exist_ok=True)

result = subprocess.run(
    ['python', '../scripts/generate_synthetic.py',
     '--output', str(raw_dir),
     '--conditions', 'no_cu', 'cu'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

## 2. Load and inspect an mzML file

In [ ]:
from src.mzml_loader import load_mzml, get_ms1_scans, get_ms2_scans

scans_no_cu, high_res = load_mzml('../data/raw/b2m_depc_no_cu.mzML')

ms1_scans = get_ms1_scans(scans_no_cu)
ms2_scans = get_ms2_scans(scans_no_cu)

print(f'Instrument: {"high-res (Orbitrap)" if high_res else "low-res (ion trap)"}')
print(f'Total scans: {len(scans_no_cu)}')
print(f'MS1 scans:   {len(ms1_scans)}')
print(f'MS2 scans:   {len(ms2_scans)}')

# Inspect the first MS1 scan
first_ms1 = next(iter(ms1_scans.values()))
print(f'\nFirst MS1 scan: RT={first_ms1["retention_time"]:.3f} min, '
      f'{len(first_ms1["peaks"])} peaks')
print(f'mz range: {first_ms1["peaks"][:,0].min():.2f} – {first_ms1["peaks"][:,0].max():.2f}')

## 3. MS1 Peak Picking

In [ ]:
from src.peak_picker import pick_all_ms1_peaks

ms1_peaks = pick_all_ms1_peaks(ms1_scans)
print(f'Picked {len(ms1_peaks)} MS1 peaks')

# Show top 10 by intensity
top10 = sorted(ms1_peaks, key=lambda p: p.intensity, reverse=True)[:10]
print(f'\n{"m/z":>10}  {"z":>3}  {"Neutral mass (Da)":>18}  {"Intensity":>12}')
for p in top10:
    print(f'{p.mz:10.4f}  {p.charge:3d}  {p.monoisotopic_mass:18.4f}  {p.intensity:12.0f}')

In [ ]:
# Plot MS1 spectrum from first scan
peaks = first_ms1['peaks']
fig, ax = plt.subplots(figsize=(10, 3))
ax.vlines(peaks[:, 0], 0, peaks[:, 1], linewidth=0.8, color='steelblue')
ax.set_xlabel('m/z')
ax.set_ylabel('Intensity')
ax.set_title('MS1 scan — synthetic β2m DEPC no-Cu(II)')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.tight_layout()
plt.show()

## 4. DEPC Event Detection

In [ ]:
from src.depc_hunter import hunt_depc_with_intensities

events_no_cu = hunt_depc_with_intensities(ms1_peaks, scans_no_cu, high_res=high_res, n_workers=2)
print(f'Confirmed DEPC events (no Cu): {len(events_no_cu)}')
print(f'Double-DEPC flagged: {sum(e.is_double_depc for e in events_no_cu)}')

# Show distribution by residue type
from collections import Counter
by_type = Counter(e.residue_type for e in events_no_cu if not e.is_double_depc)
print('\nEvents by residue type:')
for aa, n in sorted(by_type.items()):
    print(f'  {aa}: {n}')

## 5. Labeling Extent Quantification

In [ ]:
from src.labeling_quantifier import quantify_condition

# Also load and process the +Cu condition
scans_cu, _ = load_mzml('../data/raw/b2m_depc_cu.mzML')
ms1_scans_cu = get_ms1_scans(scans_cu)
ms1_peaks_cu = pick_all_ms1_peaks(ms1_scans_cu)
events_cu = hunt_depc_with_intensities(ms1_peaks_cu, scans_cu, high_res=high_res, n_workers=2)

series_no_cu = quantify_condition(events_no_cu, 'no_Cu')
series_cu    = quantify_condition(events_cu,    '+Cu')

print('Labeling extents at Cu(II)-binding histidines:')
print(f'  His13 — no Cu: {series_no_cu[13]:.3f}   +Cu: {series_cu[13]:.3f}')
print(f'  His31 — no Cu: {series_no_cu[31]:.3f}   +Cu: {series_cu[31]:.3f}')
print(f'  His51 — no Cu: {series_no_cu[51]:.3f}   +Cu: {series_cu[51]:.3f}')

In [ ]:
# Visualise labeling extents across all susceptible residues
from src.constants import B2M_RESIDUES, SUSCEPTIBLE_RESIDUES, CU_BINDING_HIS

positions = [p for p in range(1, len(B2M_RESIDUES)+1) if B2M_RESIDUES[p] in SUSCEPTIBLE_RESIDUES]
no_cu_vals = [series_no_cu.get(p, float('nan')) for p in positions]
cu_vals    = [series_cu.get(p, float('nan')) for p in positions]

fig, ax = plt.subplots(figsize=(12, 4))
x = range(len(positions))
ax.plot(x, no_cu_vals, 'o-', ms=4, label='No Cu(II)', color='steelblue')
ax.plot(x, cu_vals,    's-', ms=4, label='+Cu(II)',   color='firebrick')

# Mark His binding sites
for his_pos in CU_BINDING_HIS:
    if his_pos in positions:
        xi = positions.index(his_pos)
        ax.axvline(xi, color='orange', linestyle='--', alpha=0.6, linewidth=1)
        ax.text(xi, 0.95, f'His{his_pos}', ha='center', va='top', fontsize=8, color='darkorange')

ax.set_xticks(list(x))
ax.set_xticklabels([f'{B2M_RESIDUES[p]}{p}' for p in positions], rotation=90, fontsize=7)
ax.set_ylabel('Labeling extent')
ax.set_ylim(0, 1.05)
ax.legend()
ax.set_title('Per-residue DEPC labeling extents — synthetic β2m data')
plt.tight_layout()
plt.show()

## 6. Cu(II) Protection Analysis

In [ ]:
from src.cu_protection import compute_protection_scores
import warnings

# Wrap single replicates as single-column DataFrames
df_no_cu = series_no_cu.to_frame('r0')
df_cu    = series_cu.to_frame('r0')

with warnings.catch_warnings():
    warnings.simplefilter('ignore')  # suppress single-replicate sanity warning
    prot_df = compute_protection_scores(df_no_cu, df_cu)

# Show protection scores at His sites
print('Protection scores at His residues:')
his_positions = [p for p, aa in B2M_RESIDUES.items() if aa == 'H']
for pos in his_positions:
    if pos in prot_df.index:
        row = prot_df.loc[pos]
        print(f'  His{pos:3d}: score={row.protection_score:.3f}  p_bonf={row.p_bonferroni:.4f}  protected={row.is_protected}')

print(f'\nTotal protected residues (score>0.2, p_bonf<0.05): {prot_df.is_protected.sum()}')

In [ ]:
# Plot protection scores
valid = prot_df.dropna(subset=['protection_score'])

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['firebrick' if r else 'steelblue' for r in valid['is_protected']]
ax.bar(range(len(valid)), valid['protection_score'], color=colors, width=0.7)
ax.axhline(0.2, color='black', linestyle='--', linewidth=0.8, label='threshold (0.20)')

ax.set_xticks(range(len(valid)))
ax.set_xticklabels(
    [f'{B2M_RESIDUES[p]}{p}' for p in valid.index],
    rotation=90, fontsize=7
)
ax.set_ylabel('Protection score')
ax.set_title('Cu(II) protection scores — synthetic β2m data')
ax.legend()

# Annotate His binding sites
for i, pos in enumerate(valid.index):
    if B2M_RESIDUES[pos] == 'H' and pos in CU_BINDING_HIS:
        ax.text(i, valid.loc[pos, 'protection_score'] + 0.02, f'His{pos}',
                ha='center', fontsize=7, color='firebrick', fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Output Arrays for ML

In [ ]:
from src.array_formatter import build_condition_array, build_multi_condition_array, save_array
from src.constants import B2M_LENGTH

arr_no_cu = build_condition_array(series_no_cu)
arr_cu    = build_condition_array(series_cu)

print(f'Array shape (single condition): {arr_no_cu.shape}')
print(f'dtype: {arr_no_cu.dtype}')
print(f'Non-zero positions: {(arr_no_cu > 0).sum()} / {B2M_LENGTH}')

# Multi-condition stack
df_conditions = pd.DataFrame({'no_Cu': series_no_cu, '+Cu': series_cu})
arr_2d = build_multi_condition_array(df_conditions)
print(f'\n2D array shape: {arr_2d.shape}  (residues × conditions)')

# Save
paths = save_array(arr_2d, '../data/arrays', 'demo_labeling',
                   conditions=['no_Cu', '+Cu'])
print(f'\nSaved to: {paths["npy"]}')

In [ ]:
# Visualise the output array (heatmap)
fig, ax = plt.subplots(figsize=(14, 2.5))
im = ax.imshow(arr_2d.T, aspect='auto', cmap='viridis', vmin=0, vmax=1,
               interpolation='nearest')
ax.set_yticks([0, 1])
ax.set_yticklabels(['No Cu', '+Cu'])
ax.set_xlabel('β2m residue position (1-indexed)')
ax.set_title('DEPC labeling extent array — ready for ML')
plt.colorbar(im, ax=ax, label='Labeling extent', shrink=0.8)

# Mark His binding sites
for pos in CU_BINDING_HIS:
    ax.axvline(pos - 1, color='red', linestyle='--', alpha=0.7, linewidth=1.2)

plt.tight_layout()
plt.show()

print('Red dashed lines: His13, His31, His51 (Cu(II) binding sites)')

## 8. Full pipeline in one call

Everything above is also available via the CLI:

In [ ]:
result = subprocess.run(
    ['python', '-m', 'src.pipeline',
     '--input',  '../data/raw/',
     '--output', '../data/arrays/',
     '--mode',   'high_res'],
    capture_output=True, text=True, cwd='..'
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])